In [ ]:
!pip install qiskit-ibm-runtime qiskit qiskit_aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.8/386.8 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.8/212.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 4.8 MB/s eta 0:00:00


In [ ]:
# Load the Qiskit Runtime service
from qiskit_ibm_runtime import QiskitRuntimeService

# Load the Qiskit Runtime service

# Syntax for first saving your token.  Delete these lines after saving your credentials.
QiskitRuntimeService.save_account(channel='ibm_quantum_platform', instance = '', token='', overwrite=True, set_as_default=True)
service = QiskitRuntimeService(channel='ibm_quantum_platform')

# Load saved credentials
service = QiskitRuntimeService()


# Use the least busy backend, or uncomment the loading of a specific backend like "ibm_brisbane".
backend = service.least_busy(operational=True, simulator=False, min_num_qubits = 127)
#backend = service.backend("ibm_fez")
print(backend.name)

/usr/local/lib/python3.12/dist-packages/samplomatic/__init__.py:20: UserWarning: 
You have imported samplomatic==0.18.0 which is in 
beta development. Please expect breaking changes between 
minor versions and pin your dependencies accordingly.
  _warn_once_per_version(


ibm_fez


In [ ]:

"""
Magic-square nonlocal game on real IBM Quantum hardware using SamplerV2.
Includes explicit Readout Error Mitigation (REM) via a 16x16 calibration matrix.
Saves all raw and mitigated data to CSVs.
"""

from __future__ import annotations

import math
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import beta

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Pauli
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit_ibm_runtime.options import SamplerOptions


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
OMEGA_C = 8.0 / 9.0
ALPHA = 0.05
SHOTS_PER_SETTING = 4096

OUTDIR = Path("ibm_fez_outputs")
OUTDIR.mkdir(exist_ok=True, parents=True)

# Magic-square observables
ROWS = [["IZ", "ZI", "ZZ"], ["XI", "IX", "XX"], ["-XZ", "-ZX", "YY"]]
COLS = [list(x) for x in zip(*ROWS)]

# -----------------------------------------------------------------------------
# Linear-algebra helpers
# -----------------------------------------------------------------------------
def pauli_matrix(label: str) -> np.ndarray:
    sign = -1.0 if label.startswith("-") else 1.0
    bare = label[1:] if label.startswith("-") else label
    return sign * Pauli(bare).to_matrix()

def simultaneous_measurement_data(op_labels):
    mats = [pauli_matrix(lbl) for lbl in op_labels]
    H = mats[0] + 2.0 * mats[1] + 4.0 * mats[2]
    _, vecs = np.linalg.eigh(H)

    eig_tuples = []
    for i in range(4):
        v = vecs[:, i]
        evs = [1 if float(np.real(np.vdot(v, M @ v))) >= 0 else -1 for M in mats]
        eig_tuples.append(tuple(evs))

    order = sorted(range(4), key=lambda i: eig_tuples[i], reverse=True)
    vecs = vecs[:, order]
    eig_tuples = [eig_tuples[i] for i in order]

    for i in range(4):
        col = vecs[:, i]
        j = int(np.argmax(np.abs(col)))
        vecs[:, i] *= np.exp(-1j * np.angle(col[j]))

    return vecs, eig_tuples

ROW_BASIS = [simultaneous_measurement_data(r) for r in ROWS]
COL_BASIS = [simultaneous_measurement_data(c) for c in COLS]

# -----------------------------------------------------------------------------
# Circuit Construction
# -----------------------------------------------------------------------------
def build_magic_square_circuit(row: int, col: int) -> QuantumCircuit:
    qr = QuantumRegister(4, 'q')
    cr = ClassicalRegister(4, 'c')
    qc = QuantumCircuit(qr, cr, name=f"MS_r{row}_c{col}")

    qc.h(0)
    qc.cx(0, 2)
    qc.h(1)
    qc.cx(1, 3)
    qc.barrier()

    UA, _ = ROW_BASIS[row]
    UB, _ = COL_BASIS[col]
    qc.append(UnitaryGate(UA.conj().T, label=f"A_row{row+1}"), [0, 1])
    qc.append(UnitaryGate(UB.conj().T, label=f"B_col{col+1}"), [2, 3])
    qc.barrier()

    qc.measure(qr, cr)
    return qc

def build_calibration_circuits() -> list[QuantumCircuit]:
    cal_circuits = []
    for i in range(16):
        qr = QuantumRegister(4, 'q')
        cr = ClassicalRegister(4, 'c')
        qc = QuantumCircuit(qr, cr, name=f"Cal_{i:04b}")
        bin_str = f"{i:04b}"
        for bit_idx, bit_char in enumerate(reversed(bin_str)):
            if bit_char == '1': qc.x(bit_idx)
        qc.measure(qr, cr)
        cal_circuits.append(qc)
    return cal_circuits

# -----------------------------------------------------------------------------
# Analysis & Mitigation Helpers
# -----------------------------------------------------------------------------
def decode_round(bitstring: str, row: int, col: int) -> int:
    bs = bitstring.replace(" ", "")
    a_idx, b_idx = int(bs[2:4], 2), int(bs[0:2], 2)
    alice_tuple = ROW_BASIS[row][1][a_idx]
    bob_tuple = COL_BASIS[col][1][b_idx]
    return int(alice_tuple[col] == bob_tuple[row])

def clopper_pearson_lower(k: int, n: int) -> float:
    if n <= 0 or k == 0: return 0.0
    return float(beta.ppf(ALPHA, k, n - k + 1))

def mitigate_counts(raw_counts: dict, M_inv: np.ndarray) -> dict:
    vec = np.zeros(16)
    for bs, count in raw_counts.items():
        i = int(bs.replace(" ", ""), 2)
        vec[i] = count

    vec_mit = M_inv @ vec
    vec_mit[vec_mit < 0] = 0
    if np.sum(vec_mit) > 0:
        vec_mit = (vec_mit / np.sum(vec_mit)) * SHOTS_PER_SETTING

    return {f"{i:04b}": val for i, val in enumerate(vec_mit) if val > 0.1}


# -----------------------------------------------------------------------------
# Main Execution
# -----------------------------------------------------------------------------
def main():
    print("1. Connecting to IBM Quantum Runtime...")
    service = QiskitRuntimeService()
    backend = service.least_busy(simulator=False, operational=True, min_num_qubits=5)
    print(f"   -> Selected Backend: {backend.name}")

    print("2. Finding optimal physical mapping for 4 qubits...")
    test_qc = build_magic_square_circuit(0, 0)
    test_transpiled = transpile(test_qc, backend=backend, optimization_level=3)
    layout_dict = test_transpiled.layout.initial_layout.get_virtual_bits()
    physical_qubits = [layout_dict[q] for q in test_qc.qubits]
    print(f"   -> Locked to physical qubits: {physical_qubits}")

    print("3. Generating and Transpiling Circuits...")
    game_templates = [(r, c, build_magic_square_circuit(r, c)) for r in range(3) for c in range(3)]
    game_circuits = [qc for _, _, qc in game_templates]
    cal_circuits = build_calibration_circuits()

    transpiled_game = transpile(game_circuits, backend=backend, initial_layout=physical_qubits, optimization_level=3)
    transpiled_cal = transpile(cal_circuits, backend=backend, initial_layout=physical_qubits, optimization_level=1)
    all_circuits = transpiled_game + transpiled_cal

    print("4. Submitting Job to IBM Quantum...")
    # options = SamplerOptions()


    sampler = Sampler(mode=backend)
    job = sampler.run(all_circuits, shots=SHOTS_PER_SETTING)
    print(f"   -> Job ID: {job.job_id()}")
    print("   -> Waiting in queue... (This may take a while depending on IBM traffic)")

    result = job.result()
    print("   -> Job completed!")

    print("5. Building 16x16 Calibration Matrix...")
    M = np.zeros((16, 16))
    for j in range(16):
        counts = result[9 + j].data.c.get_counts()
        for bs, count in counts.items():
            i = int(bs.replace(" ", ""), 2)
            M[i, j] = count / float(SHOTS_PER_SETTING)
    M_inv = np.linalg.pinv(M)

    print("6. Mitigating Results & Saving Data...")
    raw_wins, raw_total = 0, 0
    mit_wins, mit_total = 0, 0

    # Lists to store detailed CSV records
    raw_counts_records = []
    mitigated_counts_records = []

    for idx, (row, col, _) in enumerate(game_templates):
        raw_counts = result[idx].data.c.get_counts()
        mit_counts = mitigate_counts(raw_counts, M_inv)

        # Save Raw Data
        for bs, count in raw_counts.items():
            win = decode_round(bs, row, col)
            raw_wins += count * win
            raw_total += count
            raw_counts_records.append({"Row": row, "Col": col, "Bitstring": bs.replace(" ", ""), "Count": count, "Win": win})

        # Save Mitigated Data
        for bs, count in mit_counts.items():
            win = decode_round(bs, row, col)
            mit_wins += count * win
            mit_total += count
            mitigated_counts_records.append({"Row": row, "Col": col, "Bitstring": bs, "Count": round(count, 3), "Win": win})

    # Compute Final Metrics
    raw_cp = clopper_pearson_lower(raw_wins, raw_total)
    mit_cp = clopper_pearson_lower(int(mit_wins), int(mit_total))

    raw_eta = max(0.0, raw_cp - OMEGA_C)
    mit_eta = max(0.0, mit_cp - OMEGA_C)

    # -------------------------------------------------------------------------
    # Save everything to CSV
    # -------------------------------------------------------------------------
    df_raw_counts = pd.DataFrame(raw_counts_records)
    df_mit_counts = pd.DataFrame(mitigated_counts_records)

    summary_data = [
        {"Configuration": f"{backend.name} (Raw)", "Average_Score": raw_wins/raw_total, "CP_Lower": raw_cp, "Eta_Min_Cert_Avg": raw_eta},
        {"Configuration": f"{backend.name} (Mitigated)", "Average_Score": mit_wins/mit_total, "CP_Lower": mit_cp, "Eta_Min_Cert_Avg": mit_eta}
    ]
    df_summary = pd.DataFrame(summary_data)

    df_raw_counts.to_csv(OUTDIR / "fez_raw_counts_real_hw.csv", index=False)
    df_mit_counts.to_csv(OUTDIR / "fez_mitigated_counts_real_hw.csv", index=False)
    df_summary.to_csv(OUTDIR / "fez_prl_real_hardware_summary.csv", index=False)

    print("\n" + "="*55)
    print("         READOUT ERROR MITIGATION RESULTS          ")
    print("="*55)
    print(f"{'Metric':<20} | {'Raw Data':<15} | {'Mitigated Data':<15}")
    print("-" * 55)
    print(f"{'Total Wins':<20} | {int(raw_wins):<15} | {int(mit_wins):<15}")
    print(f"{'Total Rounds':<20} | {int(raw_total):<15} | {int(mit_total):<15}")
    print(f"{'Average Win Rate':<20} | {(raw_wins/raw_total)*100:.2f}%{'':<9} | {(mit_wins/mit_total)*100:.2f}%")
    print(f"{'Certified η_min':<20} | {raw_eta:.5f}{'':<8} | {mit_eta:.5f}")
    print("="*55)
    print(f"\nAll data successfully saved to: {OUTDIR.resolve()}")

if __name__ == "__main__":
    main()

1. Connecting to IBM Quantum Runtime...
   -> Selected Backend: ibm_kingston
2. Finding optimal physical mapping for 4 qubits...
   -> Locked to physical qubits: [133, 11, 134, 10]
3. Generating and Transpiling Circuits...
4. Submitting Job to IBM Quantum...


/usr/local/lib/python3.12/dist-packages/qiskit_ibm_runtime/qiskit_runtime_service.py:1212: UserWarning: This instance has met its usage limit. Workloads will not run until time is made available. Check https://quantum.cloud.ibm.com/instances/crn%3Av1%3Abluemix%3Apublic%3Aquantum-computing%3Aus-east%3Aa%2F70771299a825404faa640610b8510686%3A0876f376-35a1-4351-b85d-30e0f5c5cff6%3A%3A for more details.
  warnings.warn(


   -> Job ID: d75d3jm8faus73evtujg
   -> Waiting in queue... (This may take a while depending on IBM traffic)


KeyboardInterrupt: 

In [ ]:
#!/usr/bin/env python3
"""
CHSH Nonlocal Game on real IBM Quantum hardware using SamplerV2.
Includes explicit Readout Error Mitigation (REM) via a 4x4 calibration matrix.
Saves all raw and mitigated data to CSVs.
"""

from __future__ import annotations

import math
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import beta

from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
OMEGA_C = 0.75  # Classical bound for CHSH
ALPHA = 0.05
SHOTS_PER_SETTING = 4096

OUTDIR = Path("ibm_real_hardware_chsh_outputs")
OUTDIR.mkdir(exist_ok=True, parents=True)

# -----------------------------------------------------------------------------
# Statistical & Mitigation Helpers
# -----------------------------------------------------------------------------
def clopper_pearson_lower(k: int, n: int) -> float:
    if n <= 0 or k == 0: return 0.0
    return float(beta.ppf(ALPHA, k, n - k + 1))

def eta_min_certified(lower_bound: float) -> float:
    return max(0.0, lower_bound - OMEGA_C)

def calculate_mutual_information(counts: dict) -> float:
    total = sum(counts.values())
    if total == 0: return 0.0

    p_ab, p_a, p_b = {}, {}, {}
    for bs, count in counts.items():
        bs_clean = bs.replace(" ", "")
        # Qiskit orders bits right-to-left: bit 0 is the rightmost, bit 1 is the leftmost
        a, b = bs_clean[1], bs_clean[0]
        prob = count / total
        p_ab[(a, b)] = prob
        p_a[a] = p_a.get(a, 0.0) + prob
        p_b[b] = p_b.get(b, 0.0) + prob

    mi = 0.0
    for (a, b), prob in p_ab.items():
        if prob > 0:
            mi += prob * math.log2(prob / (p_a[a] * p_b[b]))
    return mi

def mitigate_counts(raw_counts: dict, M_inv: np.ndarray) -> dict:
    vec = np.zeros(4)
    for bs, count in raw_counts.items():
        i = int(bs.replace(" ", ""), 2)
        vec[i] = count

    vec_mit = M_inv @ vec
    vec_mit[vec_mit < 0] = 0
    if np.sum(vec_mit) > 0:
        vec_mit = (vec_mit / np.sum(vec_mit)) * SHOTS_PER_SETTING

    return {f"{i:02b}": val for i, val in enumerate(vec_mit) if val > 0.1}

# -----------------------------------------------------------------------------
# Circuit Construction (CHSH)
# -----------------------------------------------------------------------------
def build_chsh_circuit(x: int, y: int) -> QuantumCircuit:
    """
    x=0: Alice measures Z, x=1: Alice measures X
    y=0: Bob measures (Z+X)/sqrt(2), y=1: Bob measures (Z-X)/sqrt(2)
    """
    qc = QuantumCircuit(2, 2, name=f"CHSH_x{x}_y{y}")

    # Create Bell State
    qc.h(0)
    qc.cx(0, 1)
    qc.barrier()

    # Alice's measurement basis (Qubit 0)
    if x == 1: qc.h(0)

    # Bob's measurement basis (Qubit 1)
    if y == 0: qc.ry(-np.pi / 4, 1)
    elif y == 1: qc.ry(np.pi / 4, 1)

    qc.barrier()
    qc.measure([0, 1], [0, 1])
    return qc

def build_calibration_circuits() -> list[QuantumCircuit]:
    """Builds 4 circuits to prepare states |00>, |01>, |10>, |11>"""
    cal_circuits = []
    for i in range(4):
        qc = QuantumCircuit(2, 2, name=f"Cal_{i:02b}")
        if i & 1: qc.x(0) # Alice bit
        if i & 2: qc.x(1) # Bob bit
        qc.measure([0, 1], [0, 1])
        cal_circuits.append(qc)
    return cal_circuits

def decode_chsh_win(bitstring: str, x: int, y: int) -> int:
    """ CHSH win condition: a XOR b == x AND y """
    bs = bitstring.replace(" ", "")
    a = int(bs[1]) # Qubit 0
    b = int(bs[0]) # Qubit 1
    return 1 if (a ^ b) == (x & y) else 0

# -----------------------------------------------------------------------------
# Main Execution
# -----------------------------------------------------------------------------
def main():
    print("1. Connecting to IBM Quantum Runtime...")
    service = QiskitRuntimeService() # Assumes Colab has valid IBMQ token
    backend = service.least_busy(simulator=False, operational=True, min_num_qubits=5)
    print(f"   -> Selected Backend: {backend.name}")

    print("2. Finding optimal physical mapping for 2 qubits...")
    test_qc = build_chsh_circuit(0, 0)
    test_transpiled = transpile(test_qc, backend=backend, optimization_level=3)
    layout_dict = test_transpiled.layout.initial_layout.get_virtual_bits()
    physical_qubits = [layout_dict[q] for q in test_qc.qubits]
    print(f"   -> Locked to physical qubits: {physical_qubits}")

    print("3. Generating and Transpiling Circuits...")
    game_templates = [(x, y, build_chsh_circuit(x, y)) for x in [0,1] for y in [0,1]]
    game_circuits = [qc for _, _, qc in game_templates]
    cal_circuits = build_calibration_circuits()

    # Transpile using the strict physical layout we found
    transpiled_game = transpile(game_circuits, backend=backend, initial_layout=physical_qubits, optimization_level=3)
    transpiled_cal = transpile(cal_circuits, backend=backend, initial_layout=physical_qubits, optimization_level=1)
    all_circuits = transpiled_game + transpiled_cal

    print("4. Submitting Job to IBM Quantum...")
    # Using mode=backend (resilience_level=0 is the default in V2)
    sampler = Sampler(mode=backend)
    job = sampler.run(all_circuits, shots=SHOTS_PER_SETTING)
    print(f"   -> Job ID: {job.job_id()}")
    print("   -> Waiting in queue... (This may take a while depending on IBM traffic)")

    result = job.result()
    print("   -> Job completed!")

    print("5. Building 4x4 Calibration Matrix...")
    M = np.zeros((4, 4))
    for j in range(4):
        # Calibration circuits start at index 4
        counts = result[4 + j].data.c.get_counts()
        for bs, count in counts.items():
            i = int(bs.replace(" ", ""), 2)
            M[i, j] = count / float(SHOTS_PER_SETTING)

    # Invert the matrix (using pseudo-inverse for numerical stability)
    M_inv = np.linalg.pinv(M)

    print("6. Mitigating Results & Saving Data...")
    raw_wins, raw_total = 0, 0
    mit_wins, mit_total = 0, 0

    raw_counts_records = []
    mitigated_counts_records = []
    mi_per_setting_raw = []
    mi_per_setting_mit = []

    for idx, (x, y, _) in enumerate(game_templates):
        raw_counts = result[idx].data.c.get_counts()
        mit_counts = mitigate_counts(raw_counts, M_inv)

        mi_per_setting_raw.append(calculate_mutual_information(raw_counts))
        mi_per_setting_mit.append(calculate_mutual_information(mit_counts))

        # Process Raw Data
        for bs, count in raw_counts.items():
            win = decode_chsh_win(bs, x, y)
            raw_wins += count * win
            raw_total += count
            raw_counts_records.append({"Setting_X": x, "Setting_Y": y, "Bitstring": bs.replace(" ", ""), "Count": count, "Win": win})

        # Process Mitigated Data
        for bs, count in mit_counts.items():
            win = decode_chsh_win(bs, x, y)
            mit_wins += count * win
            mit_total += count
            mitigated_counts_records.append({"Setting_X": x, "Setting_Y": y, "Bitstring": bs, "Count": round(count, 3), "Win": win})

    # Compute Final Metrics
    raw_cp = clopper_pearson_lower(raw_wins, raw_total)
    mit_cp = clopper_pearson_lower(int(mit_wins), int(mit_total))

    raw_eta = eta_min_certified(raw_cp)
    mit_eta = eta_min_certified(mit_cp)

    raw_mi = float(np.mean(mi_per_setting_raw))
    mit_mi = float(np.mean(mi_per_setting_mit))

    # -------------------------------------------------------------------------
    # Save everything to CSV
    # -------------------------------------------------------------------------
    df_raw_counts = pd.DataFrame(raw_counts_records)
    df_mit_counts = pd.DataFrame(mitigated_counts_records)

    summary_data = [
        {"Configuration": f"{backend.name} (Raw)", "Average_Score": raw_wins/raw_total, "CP_Lower": raw_cp, "Eta_Min_Cert_Avg": raw_eta, "Mutual_Info_Avg": raw_mi},
        {"Configuration": f"{backend.name} (Mitigated)", "Average_Score": mit_wins/mit_total, "CP_Lower": mit_cp, "Eta_Min_Cert_Avg": mit_eta, "Mutual_Info_Avg": mit_mi}
    ]
    df_summary = pd.DataFrame(summary_data)

    df_raw_counts.to_csv(OUTDIR / "raw_counts_chsh_real_hw.csv", index=False)
    df_mit_counts.to_csv(OUTDIR / "mitigated_counts_chsh_real_hw.csv", index=False)
    df_summary.to_csv(OUTDIR / "prl_chsh_real_hardware_summary.csv", index=False)

    print("\n" + "="*70)
    print("         CHSH GAME: READOUT ERROR MITIGATION RESULTS          ")
    print(f"         Classical Bound: {OMEGA_C} | Quantum Max: ~0.853      ")
    print("="*70)
    print(f"{'Metric':<25} | {'Raw Data':<15} | {'Mitigated Data':<15}")
    print("-" * 70)
    print(f"{'Total Wins':<25} | {int(raw_wins):<15} | {int(mit_wins):<15}")
    print(f"{'Average Win Rate':<25} | {(raw_wins/raw_total):.5f}{'':<8} | {(mit_wins/mit_total):.5f}")
    print(f"{'Certified η_min':<25} | {raw_eta:.5f}{'':<8} | {mit_eta:.5f}")
    print(f"{'Mean Mutual Information':<25} | {raw_mi:.5f}{'':<8} | {mit_mi:.5f}")
    print("="*70)
    print(f"\nAll data successfully saved to: {OUTDIR.resolve()}")

if __name__ == "__main__":
    main()

1. Connecting to IBM Quantum Runtime...
   -> Selected Backend: ibm_kingston
2. Finding optimal physical mapping for 2 qubits...
   -> Locked to physical qubits: [90, 89]
3. Generating and Transpiling Circuits...
4. Submitting Job to IBM Quantum...
   -> Job ID: d757slq3qcgc73frd4ug
   -> Waiting in queue... (This may take a while depending on IBM traffic)
   -> Job completed!
5. Building 4x4 Calibration Matrix...
6. Mitigating Results & Saving Data...

         CHSH GAME: READOUT ERROR MITIGATION RESULTS          
         Classical Bound: 0.75 | Quantum Max: ~0.853      
Metric                    | Raw Data        | Mitigated Data 
----------------------------------------------------------------------
Total Wins                | 13689           | 13827          
Average Win Rate          | 0.83551         | 0.84394
Certified η_min           | 0.08067         | 0.08919
Mean Mutual Information   | 0.35525         | 0.37568

All data successfully saved to: /content/ibm_real_hardware_chs

In [ ]:
#!/usr/bin/env python3
"""
Mermin-GHZ Nonlocal Game on real IBM Quantum hardware using SamplerV2.
Optimized to generate a PRL-ready summary of \eta_{min}^{cert}.
Classical Bound: 0.75
Quantum Bound: 1.00
Includes explicit 8x8 Readout Error Mitigation (REM).
"""

from __future__ import annotations

import math
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import beta

from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
OMEGA_C = 0.75  # Classical bound for GHZ/Mermin from Table I
ALPHA = 0.05
SHOTS_PER_SETTING = 4096

OUTDIR = Path("ibm_real_hardware_ghz_outputs")
OUTDIR.mkdir(exist_ok=True, parents=True)

# The 4 valid input settings for the Mermin-GHZ game (x XOR y XOR z == 0)
# 0 means measure X, 1 means measure Y.
SETTINGS = [(0,0,0), (0,1,1), (1,0,1), (1,1,0)]

# -----------------------------------------------------------------------------
# Statistical & Mitigation Helpers
# -----------------------------------------------------------------------------
def clopper_pearson_lower(k: int, n: int) -> float:
    """Eq. 49 from the manuscript: exact one-sided lower confidence limit."""
    if n <= 0 or k == 0: return 0.0
    return float(beta.ppf(ALPHA, k, n - k + 1))

def eta_min_certified(lower_bound: float) -> float:
    """Eq. 58 from the manuscript: certified benchmark dependence."""
    return max(0.0, lower_bound - OMEGA_C)

def calculate_mutual_information(counts: dict, setting_idx: int) -> float:
    """Calculates Mutual Information I(Outputs ; Inputs) for a specific setting."""
    total = sum(counts.values())
    if total == 0: return 0.0

    # For a fixed setting, the input is known, so we measure the entropy of the output.
    # To strictly calculate I(A,B,C ; X,Y,Z) across all settings, we average the entropies.
    # Here we return the Shannon entropy of the mitigated output distribution.
    entropy = 0.0
    for count in counts.values():
        if count > 0:
            p = count / total
            entropy -= p * math.log2(p)
    return entropy

def mitigate_counts(raw_counts: dict, M_inv: np.ndarray) -> dict:
    """Applies the 8x8 inverted calibration matrix to the raw counts."""
    vec = np.zeros(8)
    for bs, count in raw_counts.items():
        i = int(bs.replace(" ", ""), 2)
        vec[i] = count

    vec_mit = M_inv @ vec
    vec_mit[vec_mit < 0] = 0 # Clip negative probabilities
    if np.sum(vec_mit) > 0:
        vec_mit = (vec_mit / np.sum(vec_mit)) * SHOTS_PER_SETTING

    return {f"{i:03b}": val for i, val in enumerate(vec_mit) if val > 0.1}

# -----------------------------------------------------------------------------
# Circuit Construction (Mermin-GHZ)
# -----------------------------------------------------------------------------
def build_ghz_circuit(x: int, y: int, z: int) -> QuantumCircuit:
    """
    Inputs x, y, z in {0, 1}.
    0 -> Measure X basis (apply H)
    1 -> Measure Y basis (apply Sdg, then H)
    """
    qc = QuantumCircuit(3, 3, name=f"GHZ_{x}{y}{z}")

    # 1. Create 3-qubit GHZ State: 1/sqrt(2) * (|000> + |111>)
    qc.h(0)
    qc.cx(0, 1)
    qc.cx(1, 2)
    qc.barrier()

    # 2. Apply local measurement basis rotations
    bases = [x, y, z]
    for i, basis in enumerate(bases):
        if basis == 0:
            qc.h(i) # X basis
        elif basis == 1:
            qc.sdg(i) # Y basis
            qc.h(i)

    qc.barrier()
    qc.measure([0, 1, 2], [0, 1, 2])
    return qc

def build_calibration_circuits() -> list[QuantumCircuit]:
    """Builds 8 circuits to prepare states |000> through |111> for REM."""
    cal_circuits = []
    for i in range(8):
        qc = QuantumCircuit(3, 3, name=f"Cal_{i:03b}")
        if i & 1: qc.x(0)
        if i & 2: qc.x(1)
        if i & 4: qc.x(2)
        qc.measure([0, 1, 2], [0, 1, 2])
        cal_circuits.append(qc)
    return cal_circuits

def decode_ghz_win(bitstring: str, x: int, y: int, z: int) -> int:
    """
    Mermin-GHZ win condition:
    If x+y+z == 0 (input 000): a*b*c = +1 (Even number of 1s in bitstring)
    If x+y+z == 2 (inputs 011, 101, 110): a*b*c = -1 (Odd number of 1s in bitstring)
    """
    # Sum the bits to find parity
    num_ones = sum(1 for bit in bitstring if bit == '1')

    if (x + y + z) == 0:
        return 1 if num_ones % 2 == 0 else 0
    else:
        return 1 if num_ones % 2 == 1 else 0

# -----------------------------------------------------------------------------
# Main Execution
# -----------------------------------------------------------------------------
def main():
    print("1. Connecting to IBM Quantum Runtime...")
    service = QiskitRuntimeService() # Assumes Colab has valid IBMQ token
    backend = service.least_busy(simulator=False, operational=True, min_num_qubits=5)
    print(f"   -> Selected Backend: {backend.name}")

    print("2. Finding optimal physical mapping for 3 qubits...")
    test_qc = build_ghz_circuit(0, 0, 0)
    test_transpiled = transpile(test_qc, backend=backend, optimization_level=3)
    layout_dict = test_transpiled.layout.initial_layout.get_virtual_bits()
    physical_qubits = [layout_dict[q] for q in test_qc.qubits]
    print(f"   -> Locked to physical qubits: {physical_qubits}")

    print("3. Generating and Transpiling Circuits...")
    game_circuits = [build_ghz_circuit(x, y, z) for (x, y, z) in SETTINGS]
    cal_circuits = build_calibration_circuits()

    # Transpile using the strict contiguous physical layout we found
    transpiled_game = transpile(game_circuits, backend=backend, initial_layout=physical_qubits, optimization_level=3)
    transpiled_cal = transpile(cal_circuits, backend=backend, initial_layout=physical_qubits, optimization_level=1)
    all_circuits = transpiled_game + transpiled_cal

    print("4. Submitting Job to IBM Quantum...")
    sampler = Sampler(mode=backend)
    job = sampler.run(all_circuits, shots=SHOTS_PER_SETTING)
    print(f"   -> Job ID: {job.job_id()}")
    print("   -> Waiting in queue... (This may take a while depending on IBM traffic)")

    result = job.result()
    print("   -> Job completed!")

    print("5. Building 8x8 Calibration Matrix...")
    M = np.zeros((8, 8))
    for j in range(8):
        # Calibration circuits start at index 4 (since there are 4 game settings)
        counts = result[4 + j].data.c.get_counts()
        for bs, count in counts.items():
            i = int(bs.replace(" ", ""), 2)
            M[i, j] = count / float(SHOTS_PER_SETTING)

    # Invert the matrix
    M_inv = np.linalg.pinv(M)

    print("6. Mitigating Results & Saving Data...")
    raw_wins, raw_total = 0, 0
    mit_wins, mit_total = 0, 0

    raw_counts_records = []
    mitigated_counts_records = []
    entropies_raw = []
    entropies_mit = []

    for idx, (x, y, z) in enumerate(SETTINGS):
        raw_counts = result[idx].data.c.get_counts()
        mit_counts = mitigate_counts(raw_counts, M_inv)

        entropies_raw.append(calculate_mutual_information(raw_counts, idx))
        entropies_mit.append(calculate_mutual_information(mit_counts, idx))

        # Process Raw Data
        for bs, count in raw_counts.items():
            clean_bs = bs.replace(" ", "")
            win = decode_ghz_win(clean_bs, x, y, z)
            raw_wins += count * win
            raw_total += count
            raw_counts_records.append({"Set_X": x, "Set_Y": y, "Set_Z": z, "Bitstring": clean_bs, "Count": count, "Win": win})

        # Process Mitigated Data
        for bs, count in mit_counts.items():
            clean_bs = bs.replace(" ", "")
            win = decode_ghz_win(clean_bs, x, y, z)
            mit_wins += count * win
            mit_total += count
            mitigated_counts_records.append({"Set_X": x, "Set_Y": y, "Set_Z": z, "Bitstring": clean_bs, "Count": round(count, 3), "Win": win})

    # Compute Final Metrics (Using Clopper-Pearson from Manuscript Eq. 49)
    raw_cp = clopper_pearson_lower(raw_wins, raw_total)
    mit_cp = clopper_pearson_lower(int(mit_wins), int(mit_total))

    # Compute Certified Eta (Manuscript Eq. 58)
    raw_eta = eta_min_certified(raw_cp)
    mit_eta = eta_min_certified(mit_cp)

    mean_entropy_raw = float(np.mean(entropies_raw))
    mean_entropy_mit = float(np.mean(entropies_mit))

    # -------------------------------------------------------------------------
    # Save everything to CSV
    # -------------------------------------------------------------------------
    df_raw_counts = pd.DataFrame(raw_counts_records)
    df_mit_counts = pd.DataFrame(mitigated_counts_records)

    summary_data = [
        {"Configuration": f"{backend.name} (Raw)", "Average_Score": raw_wins/raw_total, "CP_Lower": raw_cp, "Eta_Min_Cert_Avg": raw_eta, "Output_Entropy": mean_entropy_raw},
        {"Configuration": f"{backend.name} (Mitigated)", "Average_Score": mit_wins/mit_total, "CP_Lower": mit_cp, "Eta_Min_Cert_Avg": mit_eta, "Output_Entropy": mean_entropy_mit}
    ]
    df_summary = pd.DataFrame(summary_data)

    df_raw_counts.to_csv(OUTDIR / "raw_counts_ghz_real_hw.csv", index=False)
    df_mit_counts.to_csv(OUTDIR / "mitigated_counts_ghz_real_hw.csv", index=False)
    df_summary.to_csv(OUTDIR / "prl_ghz_real_hardware_summary.csv", index=False)

    print("\n" + "="*70)
    print("       MERMIN-GHZ GAME: READOUT ERROR MITIGATION RESULTS       ")
    print(f"         Classical Bound: {OMEGA_C} | Quantum Max: 1.00      ")
    print("="*70)
    print(f"{'Metric':<25} | {'Raw Data':<15} | {'Mitigated Data':<15}")
    print("-" * 70)
    print(f"{'Total Wins':<25} | {int(raw_wins):<15} | {int(mit_wins):<15}")
    print(f"{'Average Win Rate':<25} | {(raw_wins/raw_total):.5f}{'':<8} | {(mit_wins/mit_total):.5f}")
    print(f"{'Certified η_min':<25} | {raw_eta:.5f}{'':<8} | {mit_eta:.5f}")
    print(f"{'Mean Output Entropy':<25} | {mean_entropy_raw:.5f}{'':<8} | {mean_entropy_mit:.5f}")
    print("="*70)
    print(f"\nAll data successfully saved to: {OUTDIR.resolve()}")

if __name__ == "__main__":
    main()

<>:4: SyntaxWarning: invalid escape sequence '\e'
<>:4: SyntaxWarning: invalid escape sequence '\e'
/tmp/ipykernel_7454/2366464404.py:4: SyntaxWarning: invalid escape sequence '\e'
  Optimized to generate a PRL-ready summary of \eta_{min}^{cert}.


1. Connecting to IBM Quantum Runtime...
   -> Selected Backend: ibm_kingston
2. Finding optimal physical mapping for 3 qubits...
   -> Locked to physical qubits: [91, 90, 89]
3. Generating and Transpiling Circuits...
4. Submitting Job to IBM Quantum...
   -> Job ID: d7582fa3qcgc73frdbs0
   -> Waiting in queue... (This may take a while depending on IBM traffic)
   -> Job completed!
5. Building 8x8 Calibration Matrix...
6. Mitigating Results & Saving Data...

       MERMIN-GHZ GAME: READOUT ERROR MITIGATION RESULTS       
         Classical Bound: 0.75 | Quantum Max: 1.00      
Metric                    | Raw Data        | Mitigated Data 
----------------------------------------------------------------------
Total Wins                | 15728           | 15963          
Average Win Rate          | 0.95996         | 0.97436
Certified η_min           | 0.20735         | 0.22218
Mean Output Entropy       | 2.24022         | 2.17012

All data successfully saved to: /content/ibm_real_hardware_

In [ ]:
!pip install qiskit-machine-learning scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 8.9 MB/s eta 0:00:00


In [ ]:
#!/usr/bin/env python3
"""
Explicit Readout Error Mitigated QSVM on Real IBM Quantum Hardware.
Constructs the Quantum Kernel Matrix from the ground up to evaluate
the loophole-robust certification bound (\eta_min^cert) for QML.
"""

from __future__ import annotations

import math
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import ZZFeatureMap
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit_machine_learning.datasets import ad_hoc_data

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# For ad_hoc_data, sizes are PER CLASS.
# Train: 10 per class = 20 total. Test: 5 per class = 10 total.
# Total data points = 30. Total unique kernel pairs = 465.
TRAIN_SIZE = 10
TEST_SIZE = 5
FEATURE_DIM = 2
SHOTS_PER_CIRCUIT = 4096

OUTDIR = Path("ibm_real_hardware_qml_outputs")
OUTDIR.mkdir(exist_ok=True, parents=True)

# -----------------------------------------------------------------------------
# Circuit Construction & Mitigation Helpers
# -----------------------------------------------------------------------------
def build_calibration_circuits() -> list[QuantumCircuit]:
    """Builds 4 circuits to prepare states |00>, |01>, |10>, |11>"""
    cal_circuits = []
    for i in range(4):
        qc = QuantumCircuit(2, 2, name=f"Cal_{i:02b}")
        if i & 1: qc.x(0)
        if i & 2: qc.x(1)
        qc.measure([0, 1], [0, 1])
        cal_circuits.append(qc)
    return cal_circuits

def build_kernel_circuit(x_i: np.ndarray, x_j: np.ndarray, feature_map: ZZFeatureMap) -> QuantumCircuit:
    """Creates the overlap circuit U^dagger(x_j) U(x_i) to measure the |00> probability."""
    qc1 = feature_map.assign_parameters(x_i)
    qc2 = feature_map.assign_parameters(x_j).inverse()

    qc = QuantumCircuit(2, 2)
    qc.compose(qc1, inplace=True)
    qc.compose(qc2, inplace=True)
    qc.measure([0, 1], [0, 1])
    return qc

def mitigate_counts_and_get_p00(raw_counts: dict, M_inv: np.ndarray) -> tuple[dict, float]:
    """Applies REM and returns the mitigated dictionary and the pure P(|00>) value."""
    vec = np.zeros(4)
    for bs, count in raw_counts.items():
        # Qiskit V2 bitstrings might not have spaces, but we clean them just in case
        clean_bs = bs.replace(" ", "")
        i = int(clean_bs, 2)
        vec[i] = count

    vec_mit = M_inv @ vec
    vec_mit[vec_mit < 0] = 0 # Clip negative probabilities
    total_mit = np.sum(vec_mit)

    if total_mit > 0:
        vec_mit = (vec_mit / total_mit) * SHOTS_PER_CIRCUIT

    mit_counts = {f"{i:02b}": val for i, val in enumerate(vec_mit) if val > 0.1}

    # P(|00>) is the 0th index (state '00')
    p00 = vec_mit[0] / SHOTS_PER_CIRCUIT if total_mit > 0 else 0.0
    return mit_counts, p00

def get_raw_p00(raw_counts: dict) -> float:
    """Extracts the unmitigated probability of measuring |00>."""
    # Find the count for '00' or '0 0'
    count_00 = raw_counts.get("00", raw_counts.get("0 0", 0))
    return count_00 / SHOTS_PER_CIRCUIT

# -----------------------------------------------------------------------------
# Main Execution
# -----------------------------------------------------------------------------
def main():
    print("1. Generating 'ad_hoc' Quantum Dataset...")
    # ad_hoc_data returns exactly 4 items. one_hot=False ensures sklearn compatibility.
    train_x, train_y, test_x, test_y = ad_hoc_data(
        training_size=TRAIN_SIZE,
        test_size=TEST_SIZE,
        n=FEATURE_DIM,
        gap=0.3,
        plot_data=False,
        one_hot=False  # <--- Ensures labels are 1D arrays for the SVM
    )

    # Combine data to compute the full kernel matrix efficiently
    X_all = np.vstack((train_x, test_x))
    Y_all = np.concatenate((train_y, test_y))
    num_samples = len(X_all) # Should be 30

    print("2. Establishing the Classical Loophole-Free Bound (omega_c)...")
    classical_svm = SVC(kernel='rbf', C=1.0, gamma='scale')
    classical_svm.fit(train_x, train_y)
    omega_c = classical_svm.score(test_x, test_y)
    print(f"   -> Classical SVM Accuracy (omega_c): {omega_c:.4f}")

    print("3. Connecting to IBM Quantum Runtime...")
    service = QiskitRuntimeService()
    backend = service.least_busy(simulator=False, operational=True, min_num_qubits=5)
    print(f"   -> Selected Backend: {backend.name}")

    print("4. Finding optimal physical mapping for 2 qubits...")
    feature_map = ZZFeatureMap(feature_dimension=FEATURE_DIM, reps=2, entanglement='linear')
    test_qc = build_kernel_circuit(X_all[0], X_all[0], feature_map)
    test_transpiled = transpile(test_qc, backend=backend, optimization_level=3)
    layout_dict = test_transpiled.layout.initial_layout.get_virtual_bits()
    physical_qubits = [layout_dict[q] for q in test_qc.qubits]
    print(f"   -> Locked to physical qubits: {physical_qubits}")

    print(f"5. Generating Pairwise Kernel Circuits ({num_samples} points = {num_samples*(num_samples+1)//2} circuits)...")
    kernel_circuits = []
    circuit_indices = [] # Keep track of which (i, j) pair a circuit belongs to

    for i in range(num_samples):
        for j in range(i, num_samples):
            qc = build_kernel_circuit(X_all[i], X_all[j], feature_map)
            qc.name = f"K_{i}_{j}"
            kernel_circuits.append(qc)
            circuit_indices.append((i, j))

    cal_circuits = build_calibration_circuits()

    # Transpile everything to the locked physical qubits
    transpiled_kernel = transpile(kernel_circuits, backend=backend, initial_layout=physical_qubits, optimization_level=3)
    transpiled_cal = transpile(cal_circuits, backend=backend, initial_layout=physical_qubits, optimization_level=1)

    all_circuits = transpiled_cal + transpiled_kernel

    print(f"6. Submitting {len(all_circuits)} circuits to {backend.name}...")
    sampler = Sampler(mode=backend)
    job = sampler.run(all_circuits, shots=SHOTS_PER_CIRCUIT)
    print(f"   -> Job ID: {job.job_id()}")
    print("   -> Waiting in queue... (This may take a while depending on IBM traffic)")

    result = job.result()
    print("   -> Job completed!")

    print("7. Building 4x4 Calibration Matrix...")
    M = np.zeros((4, 4))
    for j in range(4):
        counts = result[j].data.c.get_counts()
        for bs, count in counts.items():
            i = int(bs.replace(" ", ""), 2)
            M[i, j] = count / float(SHOTS_PER_CIRCUIT)
    M_inv = np.linalg.pinv(M)

    print("8. Mitigating Kernel Entries and Reconstructing Matrices...")
    K_raw = np.zeros((num_samples, num_samples))
    K_mit = np.zeros((num_samples, num_samples))

    # Calibration circuits were indices 0-3, so kernel circuits start at index 4
    for idx, (i, j) in enumerate(circuit_indices):
        raw_counts = result[4 + idx].data.c.get_counts()

        # Get probabilities
        raw_p00 = get_raw_p00(raw_counts)
        _, mit_p00 = mitigate_counts_and_get_p00(raw_counts, M_inv)

        # Populate symmetric matrices
        K_raw[i, j] = raw_p00
        K_raw[j, i] = raw_p00
        K_mit[i, j] = mit_p00
        K_mit[j, i] = mit_p00

    print("9. Evaluating SVM with Precomputed Quantum Kernels...")
    num_train = len(train_x) # Should be 20

    # Slice the full matrices into Train (20x20) and Test (10x20) chunks
    K_train_raw = K_raw[:num_train, :num_train]
    K_test_raw  = K_raw[num_train:, :num_train]

    K_train_mit = K_mit[:num_train, :num_train]
    K_test_mit  = K_mit[num_train:, :num_train]

    # Evaluate Raw Kernel
    qsvm_raw = SVC(kernel='precomputed')
    qsvm_raw.fit(K_train_raw, train_y)
    raw_acc = qsvm_raw.score(K_test_raw, test_y)

    # Evaluate Mitigated Kernel
    qsvm_mit = SVC(kernel='precomputed')
    qsvm_mit.fit(K_train_mit, train_y)
    mit_acc = qsvm_mit.score(K_test_mit, test_y)

    # Calculate PRL Certification Bounds
    eta_raw = max(0.0, raw_acc - omega_c)
    eta_mit = max(0.0, mit_acc - omega_c)

    # -------------------------------------------------------------------------
    # Save everything to CSV
    # -------------------------------------------------------------------------
    summary_data = [
        {"Configuration": f"{backend.name} (Raw)", "Accuracy": raw_acc, "Classical_Bound": omega_c, "Eta_Min_Cert_Avg": eta_raw},
        {"Configuration": f"{backend.name} (Mitigated)", "Accuracy": mit_acc, "Classical_Bound": omega_c, "Eta_Min_Cert_Avg": eta_mit}
    ]
    df_summary = pd.DataFrame(summary_data)

    # Save the matrices for potential heatmap plotting in your paper
    pd.DataFrame(K_raw).to_csv(OUTDIR / "kernel_matrix_raw.csv", index=False, header=False)
    pd.DataFrame(K_mit).to_csv(OUTDIR / "kernel_matrix_mitigated.csv", index=False, header=False)
    df_summary.to_csv(OUTDIR / "prl_qml_real_hardware_summary.csv", index=False)

    print("\n" + "="*80)
    print("      PRL SUMMARY: EXPLICITLY MITIGATED QSVM ON REAL HARDWARE      ")
    print(f"           Classical Benchmark (omega_c) established at: {omega_c:.4f}           ")
    print("="*80)
    print(f"{'Metric':<25} | {'Raw Hardware':<15} | {'Mitigated Hardware':<15}")
    print("-" * 80)
    print(f"{'SVM Test Accuracy':<25} | {raw_acc:.5f}{'':<8} | {mit_acc:.5f}")
    print(f"{'Certified η_min':<25} | {eta_raw:.5f}{'':<8} | {eta_mit:.5f}")
    print("="*80)
    print(f"\nAll data successfully saved to: {OUTDIR.resolve()}")

if __name__ == "__main__":
    main()

<>:5: SyntaxWarning: invalid escape sequence '\e'
<>:5: SyntaxWarning: invalid escape sequence '\e'
/tmp/ipykernel_7454/574591557.py:5: SyntaxWarning: invalid escape sequence '\e'
  the loophole-robust certification bound (\eta_min^cert) for QML.


1. Generating 'ad_hoc' Quantum Dataset...
2. Establishing the Classical Loophole-Free Bound (omega_c)...
   -> Classical SVM Accuracy (omega_c): 0.4000
3. Connecting to IBM Quantum Runtime...
   -> Selected Backend: ibm_kingston
4. Finding optimal physical mapping for 2 qubits...


/tmp/ipykernel_7454/574591557.py:121: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=FEATURE_DIM, reps=2, entanglement='linear')


   -> Locked to physical qubits: [10, 11]
5. Generating Pairwise Kernel Circuits (30 points = 465 circuits)...
6. Submitting 469 circuits to ibm_kingston...
   -> Job ID: d758h9fq2ubc73efk1o0
   -> Waiting in queue... (This may take a while depending on IBM traffic)
   -> Job completed!
7. Building 4x4 Calibration Matrix...
8. Mitigating Kernel Entries and Reconstructing Matrices...
9. Evaluating SVM with Precomputed Quantum Kernels...

      PRL SUMMARY: EXPLICITLY MITIGATED QSVM ON REAL HARDWARE      
           Classical Benchmark (omega_c) established at: 0.4000           
Metric                    | Raw Hardware    | Mitigated Hardware
--------------------------------------------------------------------------------
SVM Test Accuracy         | 1.00000         | 1.00000
Certified η_min           | 0.60000         | 0.60000

All data successfully saved to: /content/ibm_real_hardware_qml_outputs


In [ ]:
#!/usr/bin/env python3
"""
Explicit Readout Error Mitigated QSVM on Real IBM Quantum Hardware.
Evaluates the loophole-robust certification bound (\eta_min^cert) under both
UNBALANCED (Blocked) and BALANCED (Round-Robin) scheduling protocols.
"""

from __future__ import annotations

import math
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.svm import SVC

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import ZZFeatureMap
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit_machine_learning.datasets import ad_hoc_data

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
TRAIN_SIZE = 10
TEST_SIZE = 5
FEATURE_DIM = 2
SHOTS_PER_CIRCUIT = 4096
NUM_BINS = 6  # B=6 temporal bins (as established in the Paper for optimal drift resolution)

OUTDIR = Path("ibm_real_hardware_scheduling_outputs")
OUTDIR.mkdir(exist_ok=True, parents=True)

# -----------------------------------------------------------------------------
# Circuit Construction & Mitigation Helpers
# -----------------------------------------------------------------------------
def build_calibration_circuits() -> list[QuantumCircuit]:
    """Builds 4 circuits to prepare states |00>, |01>, |10>, |11>"""
    cal_circuits = []
    for i in range(4):
        qc = QuantumCircuit(2, 2, name=f"Cal_{i:02b}")
        if i & 1: qc.x(0)
        if i & 2: qc.x(1)
        qc.measure([0, 1], [0, 1])
        cal_circuits.append(qc)
    return cal_circuits

def build_kernel_circuit(x_i: np.ndarray, x_j: np.ndarray, feature_map: ZZFeatureMap) -> QuantumCircuit:
    """Creates the overlap circuit U^dagger(x_j) U(x_i) to measure the |00> probability."""
    qc1 = feature_map.assign_parameters(x_i)
    qc2 = feature_map.assign_parameters(x_j).inverse()
    qc = QuantumCircuit(2, 2)
    qc.compose(qc1, inplace=True)
    qc.compose(qc2, inplace=True)
    qc.measure([0, 1], [0, 1])
    return qc

def mitigate_counts_and_get_p00(raw_counts: dict, M_inv: np.ndarray, total_shots: int) -> float:
    """Applies REM to an aggregated dictionary and returns the pure P(|00>) value."""
    vec = np.zeros(4)
    for bs, count in raw_counts.items():
        clean_bs = bs.replace(" ", "")
        vec[int(clean_bs, 2)] = count

    vec_mit = M_inv @ vec
    vec_mit[vec_mit < 0] = 0 # Clip negative probabilities
    total_mit = np.sum(vec_mit)

    p00 = vec_mit[0] / total_mit if total_mit > 0 else 0.0
    return p00

def get_raw_p00(raw_counts: dict, total_shots: int) -> float:
    """Extracts the unmitigated probability of measuring |00>."""
    count_00 = raw_counts.get("00", raw_counts.get("0 0", 0))
    return count_00 / total_shots

def merge_counts(counts_list: list[dict]) -> dict:
    """Aggregates multiple count dictionaries into one (for Balanced execution)."""
    merged = {}
    for c in counts_list:
        for k, v in c.items():
            merged[k] = merged.get(k, 0) + v
    return merged

# -----------------------------------------------------------------------------
# Main Execution
# -----------------------------------------------------------------------------
def main():
    print("1. Generating 'ad_hoc' Data & Establishing Classical Bound (omega_c)...")
    train_x, train_y, test_x, test_y = ad_hoc_data(
        training_size=TRAIN_SIZE, test_size=TEST_SIZE, n=FEATURE_DIM, gap=0.3, plot_data=False, one_hot=False
    )
    X_all = np.vstack((train_x, test_x))
    Y_all = np.concatenate((train_y, test_y))
    num_samples = len(X_all)

    classical_svm = SVC(kernel='rbf', C=1.0, gamma='scale')
    classical_svm.fit(train_x, train_y)
    omega_c = classical_svm.score(test_x, test_y)
    print(f"   -> Classical SVM Accuracy (omega_c): {omega_c:.4f}")

    print("2. Connecting to IBM Quantum Runtime & Finding Mapping...")
    service = QiskitRuntimeService()
    backend = service.least_busy(simulator=False, operational=True, min_num_qubits=5)
    print(f"   -> Selected Backend: {backend.name}")

    feature_map = ZZFeatureMap(feature_dimension=FEATURE_DIM, reps=2, entanglement='linear')
    # Fix: Keep original circuit separate to iterate over its qubits
    original_test_qc = build_kernel_circuit(X_all[0], X_all[0], feature_map)
    test_qc = transpile(original_test_qc, backend=backend, optimization_level=3)
    physical_qubits = [test_qc.layout.initial_layout.get_virtual_bits()[q] for q in original_test_qc.qubits]

    print("3. Generating Pairwise Kernel Circuits...")
    kernel_circuits = []
    circuit_indices = []
    for i in range(num_samples):
        for j in range(i, num_samples):
            kernel_circuits.append(build_kernel_circuit(X_all[i], X_all[j], feature_map))
            circuit_indices.append((i, j))

    cal_circuits = build_calibration_circuits()

    # Transpile exactly once for consistency across both schedules
    transpiled_kernel = transpile(kernel_circuits, backend=backend, initial_layout=physical_qubits, optimization_level=3)
    transpiled_cal = transpile(cal_circuits, backend=backend, initial_layout=physical_qubits, optimization_level=1)
    base_circuit_list = transpiled_cal + transpiled_kernel
    num_base_circuits = len(base_circuit_list)

    print("4. Preparing Scheduling Jobs...")
    sampler = Sampler(mode=backend)

    # --- UNBALANCED SETUP ---
    # Submit the list once, 4096 shots per circuit sequentially
    print("   -> Submitting UNBALANCED Job (Blocked execution)...")
    job_unbal = sampler.run(base_circuit_list, shots=SHOTS_PER_CIRCUIT)

    # --- BALANCED SETUP ---
    # Repeat the list NUM_BINS times, interleaving the execution
    print(f"   -> Submitting BALANCED Job (Interleaved {NUM_BINS} times)...")
    shots_per_bin = SHOTS_PER_CIRCUIT // NUM_BINS
    balanced_circuit_list = base_circuit_list * NUM_BINS
    job_bal = sampler.run(balanced_circuit_list, shots=shots_per_bin)

    print(f"   -> Unbalanced Job ID: {job_unbal.job_id()}")
    print(f"   -> Balanced Job ID: {job_bal.job_id()}")
    print("   -> Waiting for execution to complete...")

    res_unbal = job_unbal.result()
    res_bal = job_bal.result()
    print("   -> Data retrieved successfully!")

    # -------------------------------------------------------------------------
    # Process Unbalanced Data
    # -------------------------------------------------------------------------
    print("5. Processing UNBALANCED Data...")
    M_unbal = np.zeros((4, 4))
    for j in range(4):
        counts = res_unbal[j].data.c.get_counts()
        for bs, count in counts.items():
            M_unbal[int(bs.replace(" ", ""), 2), j] = count / float(SHOTS_PER_CIRCUIT)
    M_inv_unbal = np.linalg.pinv(M_unbal)

    K_raw_unbal, K_mit_unbal = np.zeros((num_samples, num_samples)), np.zeros((num_samples, num_samples))
    for idx, (i, j) in enumerate(circuit_indices):
        raw_counts = res_unbal[4 + idx].data.c.get_counts()
        raw_p00 = get_raw_p00(raw_counts, SHOTS_PER_CIRCUIT)
        mit_p00 = mitigate_counts_and_get_p00(raw_counts, M_inv_unbal, SHOTS_PER_CIRCUIT)
        K_raw_unbal[i, j] = K_raw_unbal[j, i] = raw_p00
        K_mit_unbal[i, j] = K_mit_unbal[j, i] = mit_p00

    # -------------------------------------------------------------------------
    # Process Balanced Data (Requires Aggregation across Bins)
    # -------------------------------------------------------------------------
    print("6. Processing BALANCED Data (Aggregating Bins)...")
    total_balanced_shots = shots_per_bin * NUM_BINS

    # Aggregate Calibration Bins
    M_bal = np.zeros((4, 4))
    for j in range(4):
        counts_list = [res_bal[j + (bin_idx * num_base_circuits)].data.c.get_counts() for bin_idx in range(NUM_BINS)]
        merged_cal_counts = merge_counts(counts_list)
        for bs, count in merged_cal_counts.items():
            M_bal[int(bs.replace(" ", ""), 2), j] = count / float(total_balanced_shots)
    M_inv_bal = np.linalg.pinv(M_bal)

    # Aggregate Kernel Bins
    K_raw_bal, K_mit_bal = np.zeros((num_samples, num_samples)), np.zeros((num_samples, num_samples))
    for idx, (i, j) in enumerate(circuit_indices):
        circuit_offset = 4 + idx
        counts_list = [res_bal[circuit_offset + (bin_idx * num_base_circuits)].data.c.get_counts() for bin_idx in range(NUM_BINS)]
        merged_kernel_counts = merge_counts(counts_list)

        raw_p00 = get_raw_p00(merged_kernel_counts, total_balanced_shots)
        mit_p00 = mitigate_counts_and_get_p00(merged_kernel_counts, M_inv_bal, total_balanced_shots)
        K_raw_bal[i, j] = K_raw_bal[j, i] = raw_p00
        K_mit_bal[i, j] = K_mit_bal[j, i] = mit_p00

    # -------------------------------------------------------------------------
    # Evaluate SVMs & Compute Eta
    # -------------------------------------------------------------------------
    print("7. Evaluating Quantum SVMs...")
    num_train = len(train_x)

    def evaluate_qsvm(K_matrix):
        K_train = K_matrix[:num_train, :num_train]
        K_test  = K_matrix[num_train:, :num_train]
        qsvm = SVC(kernel='precomputed')
        qsvm.fit(K_train, train_y)
        acc = qsvm.score(K_test, test_y)
        return acc, max(0.0, acc - omega_c)

    acc_raw_unbal, eta_raw_unbal = evaluate_qsvm(K_raw_unbal)
    acc_mit_unbal, eta_mit_unbal = evaluate_qsvm(K_mit_unbal)
    acc_raw_bal, eta_raw_bal = evaluate_qsvm(K_raw_bal)
    acc_mit_bal, eta_mit_bal = evaluate_qsvm(K_mit_bal)

    # -------------------------------------------------------------------------
    # Summary Output
    # -------------------------------------------------------------------------
    print("\n" + "="*85)
    print("      PRL SUMMARY: QSVM SCHEDULING & PREPARATION NONSTATIONARITY      ")
    print(f"               Classical Benchmark (omega_c) established at: {omega_c:.4f}               ")
    print("="*85)
    print(f"{'Schedule & Processing':<30} | {'SVM Accuracy':<15} | {'Certified η_min':<15}")
    print("-" * 85)
    print(f"{'UNBALANCED (Blocked) - Raw':<30} | {acc_raw_unbal:.5f}{'':<8} | {eta_raw_unbal:.5f}")
    print(f"{'UNBALANCED (Blocked) - Mitig':<30} | {acc_mit_unbal:.5f}{'':<8} | {eta_mit_unbal:.5f}")
    print("-" * 85)
    print(f"{'BALANCED (Round-Robin) - Raw':<30} | {acc_raw_bal:.5f}{'':<8} | {eta_raw_bal:.5f}")
    print(f"{'BALANCED (Round-Robin) - Mitig':<30} | {acc_mit_bal:.5f}{'':<8} | {eta_mit_bal:.5f}")
    print("="*85)

if __name__ == "__main__":
    main()


<>:4: SyntaxWarning: invalid escape sequence '\e'
<>:4: SyntaxWarning: invalid escape sequence '\e'
/tmp/ipykernel_2538/62112969.py:4: SyntaxWarning: invalid escape sequence '\e'
  Evaluates the loophole-robust certification bound (\eta_min^cert) under both


1. Generating 'ad_hoc' Data & Establishing Classical Bound (omega_c)...
   -> Classical SVM Accuracy (omega_c): 0.6000
2. Connecting to IBM Quantum Runtime & Finding Mapping...
   -> Selected Backend: ibm_fez


/tmp/ipykernel_2538/62112969.py:106: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=FEATURE_DIM, reps=2, entanglement='linear')


3. Generating Pairwise Kernel Circuits...
4. Preparing Scheduling Jobs...
   -> Submitting UNBALANCED Job (Blocked execution)...


/usr/local/lib/python3.12/dist-packages/qiskit_ibm_runtime/qiskit_runtime_service.py:1212: UserWarning: This instance has met its usage limit. Workloads will not run until time is made available. Check https://quantum.cloud.ibm.com/instances/crn%3Av1%3Abluemix%3Apublic%3Aquantum-computing%3Aus-east%3Aa%2F70771299a825404faa640610b8510686%3A0876f376-35a1-4351-b85d-30e0f5c5cff6%3A%3A for more details.
  warnings.warn(


   -> Submitting BALANCED Job (Interleaved 6 times)...
   -> Unbalanced Job ID: d764i468faus73f0qha0
   -> Balanced Job ID: d764i75bjrds73ed54vg
   -> Waiting for execution to complete...
   -> Data retrieved successfully!
5. Processing UNBALANCED Data...
6. Processing BALANCED Data (Aggregating Bins)...
7. Evaluating Quantum SVMs...

      PRL SUMMARY: QSVM SCHEDULING & PREPARATION NONSTATIONARITY      
               Classical Benchmark (omega_c) established at: 0.6000               
Schedule & Processing          | SVM Accuracy    | Certified η_min
-------------------------------------------------------------------------------------
UNBALANCED (Blocked) - Raw     | 1.00000         | 0.40000
UNBALANCED (Blocked) - Mitig   | 1.00000         | 0.40000
-------------------------------------------------------------------------------------
BALANCED (Round-Robin) - Raw   | 1.00000         | 0.40000
BALANCED (Round-Robin) - Mitig | 1.00000         | 0.40000


Scheduled vs Unscheduled

In [ ]:
#!/usr/bin/env python3
"""
Unified 6-bin macro-round scheduler for CHSH, GHZ, and Magic Square
on real IBM Quantum hardware using SamplerV2.

What this does:
- Uses a total shot budget N per setting
- Splits N into 6 bins
- Treats each bin as one macro-round
- Supports two temporal schedules:
    1) bin_interleaved   : all settings for bin 1, then all settings for bin 2, ...
    2) setting_clustered : all bins for setting 1, then all bins for setting 2, ...
- Computes:
    * average-score certification
    * 6-bin product-style certification from simultaneous per-bin lower bounds

Output:
- <schedule_name>_bin_setting_results.csv
- <schedule_name>_bin_summary.csv
- <schedule_name>_summary.csv
- combined_summary.csv
"""

from __future__ import annotations

import math
from pathlib import Path
from typing import Any, Callable, Iterable

import numpy as np
import pandas as pd
from scipy.stats import beta

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Pauli
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler


# =============================================================================
# USER CONFIGURATION
# =============================================================================
GAME = "magic_square"                  # "chsh", "ghz", "magic_square"
N_SHOTS_PER_SETTING = 4096
NUM_BINS = 6
ALPHA = 0.05
CAL_SHOTS = 4096

# Run one or both schedules
RUN_BIN_INTERLEAVED = True
RUN_SETTING_CLUSTERED = True

OUTDIR = Path("scheduled_bin_outputs")
OUTDIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# SHARED HELPERS
# =============================================================================
def split_into_bins(n: int, num_bins: int) -> list[int]:
    """Split integer n into num_bins nearly equal parts."""
    q, r = divmod(n, num_bins)
    return [q + 1 if i < r else q for i in range(num_bins)]


def clopper_pearson_lower(k: int, n: int, alpha: float) -> float:
    """Exact one-sided Clopper-Pearson lower bound."""
    if n <= 0 or k == 0:
        return 0.0
    return float(beta.ppf(alpha, k, n - k + 1))


def eta_min_cert_avg(lower_bound: float, omega_c: float) -> float:
    return max(0.0, lower_bound - omega_c)


def eta_min_cert_block_from_bins(bin_lowers: list[float], omega_c: float) -> tuple[float, float]:
    """
    Treat 6 bins as 6 macro-rounds.
    If L_b are simultaneous lower bounds for each bin score, define
        L_prod = prod_b L_b
    and certify
        eta = max(0, L_prod^(1/6) - omega_c)
    """
    if len(bin_lowers) == 0:
        return 0.0, 0.0
    l_prod = float(np.prod(bin_lowers))
    eta = max(0.0, l_prod ** (1.0 / len(bin_lowers)) - omega_c)
    return l_prod, eta


def normalize_counts_to_shots(raw_counts: dict[str, float], target_shots: int) -> dict[str, float]:
    """Normalize a quasi-distribution back to target_shots."""
    total = float(sum(raw_counts.values()))
    if total <= 0:
        return {}
    out = {}
    for bs, val in raw_counts.items():
        if val > 0:
            out[bs] = target_shots * float(val) / total
    return out


# =============================================================================
# CHSH
# =============================================================================
def setup_chsh():
    omega_c = 0.75
    settings = [(0, 0), (0, 1), (1, 0), (1, 1)]

    def build_circuit(ctx: tuple[int, int]) -> QuantumCircuit:
        x, y = ctx
        qc = QuantumCircuit(2, 2, name=f"CHSH_x{x}_y{y}")

        qc.h(0)
        qc.cx(0, 1)
        qc.barrier()

        if x == 1:
            qc.h(0)

        if y == 0:
            qc.ry(-np.pi / 4, 1)
        elif y == 1:
            qc.ry(np.pi / 4, 1)

        qc.barrier()
        qc.measure([0, 1], [0, 1])
        return qc

    def build_calibration_circuits() -> list[QuantumCircuit]:
        circs = []
        for i in range(4):
            qc = QuantumCircuit(2, 2, name=f"Cal_{i:02b}")
            if i & 1:
                qc.x(0)
            if i & 2:
                qc.x(1)
            qc.measure([0, 1], [0, 1])
            circs.append(qc)
        return circs

    def decode_win(bitstring: str, ctx: tuple[int, int]) -> int:
        bs = bitstring.replace(" ", "")
        x, y = ctx
        a = int(bs[1])  # qubit 0
        b = int(bs[0])  # qubit 1
        return 1 if (a ^ b) == (x & y) else 0

    def mitigate_counts(raw_counts: dict[str, int], m_inv: np.ndarray, target_shots: int) -> dict[str, float]:
        vec = np.zeros(4)
        for bs, count in raw_counts.items():
            i = int(bs.replace(" ", ""), 2)
            vec[i] = count
        vec_mit = m_inv @ vec
        vec_mit[vec_mit < 0] = 0
        out = {f"{i:02b}": float(val) for i, val in enumerate(vec_mit) if val > 1e-12}
        return normalize_counts_to_shots(out, target_shots)

    return omega_c, settings, build_circuit, build_calibration_circuits, decode_win, mitigate_counts


# =============================================================================
# GHZ
# =============================================================================
def setup_ghz():
    omega_c = 0.75
    settings = [(0, 0, 0), (0, 1, 1), (1, 0, 1), (1, 1, 0)]

    def build_circuit(ctx: tuple[int, int, int]) -> QuantumCircuit:
        x, y, z = ctx
        qc = QuantumCircuit(3, 3, name=f"GHZ_{x}{y}{z}")

        qc.h(0)
        qc.cx(0, 1)
        qc.cx(1, 2)
        qc.barrier()

        for i, basis in enumerate([x, y, z]):
            if basis == 0:
                qc.h(i)
            elif basis == 1:
                qc.sdg(i)
                qc.h(i)

        qc.barrier()
        qc.measure([0, 1, 2], [0, 1, 2])
        return qc

    def build_calibration_circuits() -> list[QuantumCircuit]:
        circs = []
        for i in range(8):
            qc = QuantumCircuit(3, 3, name=f"Cal_{i:03b}")
            if i & 1:
                qc.x(0)
            if i & 2:
                qc.x(1)
            if i & 4:
                qc.x(2)
            qc.measure([0, 1, 2], [0, 1, 2])
            circs.append(qc)
        return circs

    def decode_win(bitstring: str, ctx: tuple[int, int, int]) -> int:
        x, y, z = ctx
        num_ones = sum(1 for bit in bitstring if bit == "1")
        if (x + y + z) == 0:
            return 1 if num_ones % 2 == 0 else 0
        return 1 if num_ones % 2 == 1 else 0

    def mitigate_counts(raw_counts: dict[str, int], m_inv: np.ndarray, target_shots: int) -> dict[str, float]:
        vec = np.zeros(8)
        for bs, count in raw_counts.items():
            i = int(bs.replace(" ", ""), 2)
            vec[i] = count
        vec_mit = m_inv @ vec
        vec_mit[vec_mit < 0] = 0
        out = {f"{i:03b}": float(val) for i, val in enumerate(vec_mit) if val > 1e-12}
        return normalize_counts_to_shots(out, target_shots)

    return omega_c, settings, build_circuit, build_calibration_circuits, decode_win, mitigate_counts


# =============================================================================
# MAGIC SQUARE
# =============================================================================
ROWS = [["IZ", "ZI", "ZZ"], ["XI", "IX", "XX"], ["-XZ", "-ZX", "YY"]]
COLS = [list(x) for x in zip(*ROWS)]


def pauli_matrix(label: str) -> np.ndarray:
    sign = -1.0 if label.startswith("-") else 1.0
    bare = label[1:] if label.startswith("-") else label
    return sign * Pauli(bare).to_matrix()


def simultaneous_measurement_data(op_labels: list[str]) -> tuple[np.ndarray, list[tuple[int, int, int]]]:
    mats = [pauli_matrix(lbl) for lbl in op_labels]
    h = mats[0] + 2.0 * mats[1] + 4.0 * mats[2]
    _, vecs = np.linalg.eigh(h)

    eig_tuples: list[tuple[int, int, int]] = []
    for i in range(4):
        v = vecs[:, i]
        evs = tuple(1 if float(np.real(np.vdot(v, m @ v))) >= 0 else -1 for m in mats)
        eig_tuples.append(evs)

    order = sorted(range(4), key=lambda i: eig_tuples[i], reverse=True)
    vecs = vecs[:, order]
    eig_tuples = [eig_tuples[i] for i in order]

    for i in range(4):
        col = vecs[:, i]
        j = int(np.argmax(np.abs(col)))
        vecs[:, i] *= np.exp(-1j * np.angle(col[j]))

    return vecs, eig_tuples


ROW_BASIS = [simultaneous_measurement_data(r) for r in ROWS]
COL_BASIS = [simultaneous_measurement_data(c) for c in COLS]


def setup_magic_square():
    omega_c = 8.0 / 9.0
    settings = [(r, c) for r in range(3) for c in range(3)]

    def build_circuit(ctx: tuple[int, int]) -> QuantumCircuit:
        row, col = ctx
        qr = QuantumRegister(4, "q")
        cr = ClassicalRegister(4, "c")
        qc = QuantumCircuit(qr, cr, name=f"MS_r{row}_c{col}")

        qc.h(0)
        qc.cx(0, 2)
        qc.h(1)
        qc.cx(1, 3)
        qc.barrier()

        ua, _ = ROW_BASIS[row]
        ub, _ = COL_BASIS[col]
        qc.append(UnitaryGate(ua.conj().T, label=f"A_row{row+1}"), [0, 1])
        qc.append(UnitaryGate(ub.conj().T, label=f"B_col{col+1}"), [2, 3])
        qc.barrier()

        qc.measure(qr, cr)
        return qc

    def build_calibration_circuits() -> list[QuantumCircuit]:
        circs = []
        for i in range(16):
            qr = QuantumRegister(4, "q")
            cr = ClassicalRegister(4, "c")
            qc = QuantumCircuit(qr, cr, name=f"Cal_{i:04b}")
            bin_str = f"{i:04b}"
            for bit_idx, bit_char in enumerate(reversed(bin_str)):
                if bit_char == "1":
                    qc.x(bit_idx)
            qc.measure(qr, cr)
            circs.append(qc)
        return circs

    def decode_win(bitstring: str, ctx: tuple[int, int]) -> int:
        row, col = ctx
        bs = bitstring.replace(" ", "")
        a_idx = int(bs[2:4], 2)
        b_idx = int(bs[0:2], 2)
        alice_tuple = ROW_BASIS[row][1][a_idx]
        bob_tuple = COL_BASIS[col][1][b_idx]
        return int(alice_tuple[col] == bob_tuple[row])

    def mitigate_counts(raw_counts: dict[str, int], m_inv: np.ndarray, target_shots: int) -> dict[str, float]:
        vec = np.zeros(16)
        for bs, count in raw_counts.items():
            i = int(bs.replace(" ", ""), 2)
            vec[i] = count
        vec_mit = m_inv @ vec
        vec_mit[vec_mit < 0] = 0
        out = {f"{i:04b}": float(val) for i, val in enumerate(vec_mit) if val > 1e-12}
        return normalize_counts_to_shots(out, target_shots)

    return omega_c, settings, build_circuit, build_calibration_circuits, decode_win, mitigate_counts


# =============================================================================
# SCHEDULE EXECUTION
# =============================================================================
def build_jobspec(
    settings: list[Any],
    shots_per_bin: list[int],
    schedule_name: str,
) -> list[dict[str, Any]]:
    jobspec: list[dict[str, Any]] = []

    if schedule_name == "bin_interleaved":
        for b, n_b in enumerate(shots_per_bin):
            for s_idx, setting in enumerate(settings):
                jobspec.append(
                    {
                        "schedule_name": schedule_name,
                        "bin_id": b,
                        "setting_id": s_idx,
                        "setting": setting,
                        "shots": n_b,
                    }
                )
    elif schedule_name == "setting_clustered":
        for s_idx, setting in enumerate(settings):
            for b, n_b in enumerate(shots_per_bin):
                jobspec.append(
                    {
                        "schedule_name": schedule_name,
                        "bin_id": b,
                        "setting_id": s_idx,
                        "setting": setting,
                        "shots": n_b,
                    }
                )
    else:
        raise ValueError(f"Unknown schedule_name = {schedule_name}")

    return jobspec


def build_calibration_matrix(
    sampler: Sampler,
    backend,
    build_calibration_circuits: Callable[[], list[QuantumCircuit]],
    initial_layout: list[int],
) -> np.ndarray:
    cal_circuits = build_calibration_circuits()
    transpiled_cal = transpile(
        cal_circuits,
        backend=backend,
        initial_layout=initial_layout,
        optimization_level=1,
    )

    cal_pubs = [(qc, None, CAL_SHOTS) for qc in transpiled_cal]
    cal_job = sampler.run(cal_pubs)
    print(f"Calibration job id: {cal_job.job_id()}")
    cal_result = cal_job.result()

    dim = len(cal_circuits)
    m = np.zeros((dim, dim))
    for j in range(dim):
        counts = cal_result[j].data.c.get_counts()
        for bs, count in counts.items():
            i = int(bs.replace(" ", ""), 2)
            m[i, j] = count / float(CAL_SHOTS)

    return np.linalg.pinv(m)


def run_schedule(
    *,
    backend,
    sampler: Sampler,
    settings: list[Any],
    omega_c: float,
    build_circuit: Callable[[Any], QuantumCircuit],
    decode_win: Callable[[str, Any], int],
    mitigate_counts: Callable[[dict[str, int], np.ndarray, int], dict[str, float]],
    m_inv: np.ndarray,
    initial_layout: list[int],
    schedule_name: str,
    shots_per_bin: list[int],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    jobspec = build_jobspec(settings, shots_per_bin, schedule_name)
    circuits = [build_circuit(spec["setting"]) for spec in jobspec]

    transpiled_game = transpile(
        circuits,
        backend=backend,
        initial_layout=initial_layout,
        optimization_level=3,
    )

    pubs = [(transpiled_game[i], None, jobspec[i]["shots"]) for i in range(len(transpiled_game))]
    job = sampler.run(pubs)
    print(f"{schedule_name} job id: {job.job_id()}")
    result = job.result()

    records = []
    total_wins_raw = 0
    total_rounds_raw = 0
    total_wins_mit = 0.0
    total_rounds_mit = 0.0

    for i, spec in enumerate(jobspec):
        raw_counts = result[i].data.c.get_counts()
        mit_counts = mitigate_counts(raw_counts, m_inv, spec["shots"])

        wins_raw = 0
        rounds_raw = 0
        for bs, count in raw_counts.items():
            clean_bs = bs.replace(" ", "")
            win = decode_win(clean_bs, spec["setting"])
            wins_raw += count * win
            rounds_raw += count

        wins_mit = 0.0
        rounds_mit = 0.0
        for bs, count in mit_counts.items():
            clean_bs = bs.replace(" ", "")
            win = decode_win(clean_bs, spec["setting"])
            wins_mit += count * win
            rounds_mit += count

        total_wins_raw += wins_raw
        total_rounds_raw += rounds_raw
        total_wins_mit += wins_mit
        total_rounds_mit += rounds_mit

        records.append(
            {
                "schedule_name": schedule_name,
                "bin_id": spec["bin_id"],
                "setting_id": spec["setting_id"],
                "setting": str(spec["setting"]),
                "shots": rounds_raw,
                "wins_raw": wins_raw,
                "score_raw": wins_raw / rounds_raw if rounds_raw > 0 else np.nan,
                "wins_mit": wins_mit,
                "score_mit": wins_mit / rounds_mit if rounds_mit > 0 else np.nan,
            }
        )

    df = pd.DataFrame(records)
    df.to_csv(OUTDIR / f"{schedule_name}_bin_setting_results.csv", index=False)

    # ---------------- average-score certification ----------------
    l_avg_raw = clopper_pearson_lower(int(total_wins_raw), int(total_rounds_raw), ALPHA)
    eta_avg_raw = eta_min_cert_avg(l_avg_raw, omega_c)

    # For mitigated averages, use conservative integer cast for CP input
    l_avg_mit = clopper_pearson_lower(int(round(total_wins_mit)), int(round(total_rounds_mit)), ALPHA)
    eta_avg_mit = eta_min_cert_avg(l_avg_mit, omega_c)

    # ---------------- bin-level macro-round lower bounds ----------------
    alpha_bin = ALPHA / NUM_BINS

    bin_rows = []
    bin_lowers_raw: list[float] = []
    bin_lowers_mit: list[float] = []

    for b, g in df.groupby("bin_id", sort=True):
        k_raw = int(g["wins_raw"].sum())
        n_raw = int(g["shots"].sum())
        qhat_raw = k_raw / n_raw
        l_raw = clopper_pearson_lower(k_raw, n_raw, alpha_bin)

        k_mit = float(g["wins_mit"].sum())
        n_mit = float(g["shots"].sum())
        qhat_mit = k_mit / n_mit if n_mit > 0 else np.nan
        l_mit = clopper_pearson_lower(int(round(k_mit)), int(round(n_mit)), alpha_bin)

        bin_lowers_raw.append(l_raw)
        bin_lowers_mit.append(l_mit)

        bin_rows.append(
            {
                "schedule_name": schedule_name,
                "bin_id": int(b),
                "total_wins_bin_raw": k_raw,
                "total_shots_bin": n_raw,
                "score_bin_raw": qhat_raw,
                "cp_lower_bin_raw": l_raw,
                "total_wins_bin_mit": k_mit,
                "score_bin_mit": qhat_mit,
                "cp_lower_bin_mit": l_mit,
            }
        )

    df_bins = pd.DataFrame(bin_rows)
    df_bins.to_csv(OUTDIR / f"{schedule_name}_bin_summary.csv", index=False)

    l_prod_raw, eta_block_raw = eta_min_cert_block_from_bins(bin_lowers_raw, omega_c)
    l_prod_mit, eta_block_mit = eta_min_cert_block_from_bins(bin_lowers_mit, omega_c)

    summary = pd.DataFrame(
        [
            {
                "schedule_name": schedule_name,
                "average_score_raw": total_wins_raw / total_rounds_raw,
                "cp_lower_avg_raw": l_avg_raw,
                "eta_min_cert_avg_raw": eta_avg_raw,
                "product_lower_from_bins_raw": l_prod_raw,
                "eta_min_cert_block_6bins_raw": eta_block_raw,
                "average_score_mit": total_wins_mit / total_rounds_mit if total_rounds_mit > 0 else np.nan,
                "cp_lower_avg_mit": l_avg_mit,
                "eta_min_cert_avg_mit": eta_avg_mit,
                "product_lower_from_bins_mit": l_prod_mit,
                "eta_min_cert_block_6bins_mit": eta_block_mit,
            }
        ]
    )
    summary.to_csv(OUTDIR / f"{schedule_name}_summary.csv", index=False)

    return df, df_bins, summary


# =============================================================================
# MAIN
# =============================================================================
def main():
    if GAME == "chsh":
        omega_c, settings, build_circuit, build_calibration_circuits, decode_win, mitigate_counts = setup_chsh()
    elif GAME == "ghz":
        omega_c, settings, build_circuit, build_calibration_circuits, decode_win, mitigate_counts = setup_ghz()
    elif GAME == "magic_square":
        omega_c, settings, build_circuit, build_calibration_circuits, decode_win, mitigate_counts = setup_magic_square()
    else:
        raise ValueError(f"Unknown GAME = {GAME}")

    shots_per_bin = split_into_bins(N_SHOTS_PER_SETTING, NUM_BINS)

    print(f"Game: {GAME}")
    print(f"Settings: {settings}")
    print(f"Classical bound omega_c = {omega_c}")
    print(f"N shots per setting = {N_SHOTS_PER_SETTING}")
    print(f"6-bin split = {shots_per_bin}")

    # Connect
    print("Connecting to IBM Quantum Runtime...")
    service = QiskitRuntimeService()
    backend = service.least_busy(simulator=False, operational=True, min_num_qubits=5)
    print(f"Selected backend: {backend.name}")

    # Lock layout from first game circuit
    test_qc = build_circuit(settings[0])
    test_transpiled = transpile(test_qc, backend=backend, optimization_level=3)
    layout_dict = test_transpiled.layout.initial_layout.get_virtual_bits()
    physical_qubits = [layout_dict[q] for q in test_qc.qubits]
    print(f"Locked physical qubits: {physical_qubits}")

    sampler = Sampler(mode=backend)

    print("Building calibration matrix...")
    m_inv = build_calibration_matrix(
        sampler=sampler,
        backend=backend,
        build_calibration_circuits=build_calibration_circuits,
        initial_layout=physical_qubits,
    )

    all_summaries = []

    if RUN_BIN_INTERLEAVED:
        print("\nRunning bin_interleaved schedule...")
        _, _, s = run_schedule(
            backend=backend,
            sampler=sampler,
            settings=settings,
            omega_c=omega_c,
            build_circuit=build_circuit,
            decode_win=decode_win,
            mitigate_counts=mitigate_counts,
            m_inv=m_inv,
            initial_layout=physical_qubits,
            schedule_name="bin_interleaved",
            shots_per_bin=shots_per_bin,
        )
        all_summaries.append(s)

    if RUN_SETTING_CLUSTERED:
        print("\nRunning setting_clustered schedule...")
        _, _, s = run_schedule(
            backend=backend,
            sampler=sampler,
            settings=settings,
            omega_c=omega_c,
            build_circuit=build_circuit,
            decode_win=decode_win,
            mitigate_counts=mitigate_counts,
            m_inv=m_inv,
            initial_layout=physical_qubits,
            schedule_name="setting_clustered",
            shots_per_bin=shots_per_bin,
        )
        all_summaries.append(s)

    if all_summaries:
        combined = pd.concat(all_summaries, ignore_index=True)
        combined.to_csv(OUTDIR / "combined_summary.csv", index=False)

        print("\nCombined summary:")
        print(combined.to_string(index=False))

    print(f"\nAll outputs saved to: {OUTDIR.resolve()}")


if __name__ == "__main__":
    main()

Game: magic_square
Settings: [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)]
Classical bound omega_c = 0.8888888888888888
N shots per setting = 4096
6-bin split = [683, 683, 683, 683, 682, 682]
Connecting to IBM Quantum Runtime...
Selected backend: ibm_fez
Locked physical qubits: [114, 23, 115, 22]
Building calibration matrix...
Calibration job id: d7eh9m95a5qc73dqv070

Running bin_interleaved schedule...
bin_interleaved job id: d7eh9sp5a5qc73dqv0fg

Running setting_clustered schedule...
setting_clustered job id: d7eha6e5nvhs73a6ondg

Combined summary:
    schedule_name  average_score_raw  cp_lower_avg_raw  eta_min_cert_avg_raw  product_lower_from_bins_raw  eta_min_cert_block_6bins_raw  average_score_mit  cp_lower_avg_mit  eta_min_cert_avg_mit  product_lower_from_bins_mit  eta_min_cert_block_6bins_mit
  bin_interleaved           0.752930          0.749206                   0.0                     0.163453                           0.0           0.760909        